In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הגדרות ברירת מחדל ואפשרויות תצורה של ה-Transpiler


<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח בשימוש בדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלה או חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

מעגלים אבסטרקטיים חייבים לעבור טרנספילציה מפני ש-QPU-ים כוללים קבוצה מוגבלת של שערי בסיס ואינם יכולים לבצע פעולות שרירותיות. תפקיד ה-Transpiler הוא לשנות מעגלים שרירותיים כך שיוכלו לרוץ על QPU מסוים. הדבר נעשה על ידי תרגום המעגלים לשערי הבסיס הנתמכים, ועל ידי הוספת שערי SWAP לפי הצורך, כך שהקישוריות של המעגל תתאים לזו של ה-QPU.

כפי שמוסבר ב[טרנספילציה עם pass managers](transpile-with-pass-managers), ניתן ליצור [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) באמצעות הפונקציה [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) ולהעביר מעגל או רשימת מעגלים למתודת [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) שלו כדי לבצע טרנספילציה עליהם. ניתן לקרוא ל-`generate_preset_pass_manager` עם רמת האופטימיזציה וה-Backend בלבד ולהשתמש בברירות המחדל לשאר האפשרויות, או להעביר ארגומנטים נוספים לפונקציה לכיוונון עדין של הטרנספילציה.

## שימוש בסיסי ללא פרמטרים
בדוגמה זו, אנו מעבירים מעגל ו-QPU יעד ל-Transpiler מבלי לציין פרמטרים נוספים.

צור מעגל וצפה בתוצאה:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

בצע טרנספילציה על המעגל וצפה בתוצאה:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## כל הפרמטרים הזמינים
להלן כל הפרמטרים הזמינים לפונקציה [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). ישנן שתי קבוצות של ארגומנטים: אלה המתארים את יעד הקומפילציה, ואלה המשפיעים על אופן פעולת ה-Transpiler.

כל הפרמטרים פרט ל-`optimization_level` הם אופציונליים. לפרטים מלאים, ראה את [תיעוד ה-API של ה-Transpiler](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - כמה אופטימיזציה לבצע על המעגלים. מספר שלם בטווח (0-3). רמות גבוהות יותר מייצרות מעגלים מאופטמים יותר, על חשבון זמן טרנספילציה ארוך יותר. ראה [הגדרת רמת אופטימיזציה של ה-Transpiler](set-optimization) לפרטים נוספים.

### פרמטרים המשמשים לתיאור יעד הקומפילציה:
ארגומנטים אלה מתארים את ה-QPU היעד להרצת המעגל, כולל מידע כגון מפת הצימוד של ה-QPU (המתארת את הקישוריות של ה-Qubit-ים), שערי הבסיס הנתמכים על ידי ה-QPU, ושיעורי השגיאה של השערים.

פרמטרים רבים מאלה מתוארים בפירוט ב[פרמטרים נפוצים לטרנספילציה](common-parameters).

<details>
  <summary>
    **פרמטרי QPU (`Backend`)**
  </summary>

**פרמטרי Backend** - אם תציין `backend`, אינך צריך לציין `target` או אפשרויות backend אחרות. כמו כן, אם תציין `target`, אינך צריך לציין `backend` או אפשרויות backend אחרות.
- `backend` (Backend) - אם הוגדר, ה-Transpiler מקמפל את מעגל הקלט למכשיר זה. אם הוגדרה אפשרות אחרת שמשפיעה על הגדרות אלה, כגון `coupling_map`, היא עוקפת את ההגדרות מ-`backend`.
- `target` (Target) - יעד Transpiler של Backend. בדרך כלל זה מצוין כחלק מארגומנט ה-backend, אך אם בנית ידנית אובייקט Target, ניתן לציין אותו כאן. זה עוקף את היעד מ-`backend`.
- `backend_properties` (BackendProperties) - מאפיינים שמוחזרים על ידי QPU, כולל מידע על שגיאות שערים, שגיאות קריאה, זמני קוהרנטיות של Qubit-ים, וכן הלאה. מצא QPU שמספק מידע זה על ידי הרצת `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - הגבלה אופציונלית של חומרת הבקרה על רזולוציית זמן ההוראות. מידע זה מסופק על ידי תצורת ה-QPU. אם ל-QPU אין הגבלה על הקצאת זמן ההוראות, `timing_constraints` הוא `None` ולא בוצעת שום התאמה. QPU עשוי לדווח על קבוצת הגבלות, כלומר:
    - `granularity`: ערך שלם המייצג את רזולוציית שער הפולס המינימלית ביחידות של dt. לשער פולס מוגדר-משתמש צריכה להיות משך זמן שהוא כפולה של ערך הגרנולריות הזה.
    - `min_length`: ערך שלם המייצג את אורך שער הפולס המינימלי ביחידות של dt. שער פולס מוגדר-משתמש צריך להיות ארוך מאורך זה.
    - `pulse_alignment`: ערך שלם המייצג רזולוציית זמן של זמן התחלת הוראת שער. הוראות שער צריכות להתחיל בזמן שהוא כפולה של ערך זה.
    - `acquire_alignment`: ערך שלם המייצג רזולוציית זמן של זמן התחלת הוראת מדידה. הוראת מדידה צריכה להתחיל בזמן שהוא כפולה של ערך זה.
</details>

<details>
  <summary>
    **פרמטרי פריסה וטופולוגיה**
  </summary>

- `basis_gates` (List[str] | None) - רשימת שמות שערי בסיס לפריסה אליהם. לדוגמה ['u1', 'u2', 'u3', 'cx']. אם `None`, אל תבצע פריסה.
- `coupling_map` (CouplingMap | List[List[int]]) - מפת צימוד מכוונת (אפשרי מותאמת אישית) ליעד במיפוי. אם מפת הצימוד סימטרית, שני הכיוונים צריכים להיות מצוינים. הפורמטים הבאים נתמכים:
    - מופע CouplingMap
    - List - חייב להינתן כמטריצת שכנות, כאשר כל רשומה מציינת את כל האינטראקציות הדו-Qubit המכוונות הנתמכות על ידי ה-QPU. לדוגמה: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - מיפוי של פעולות מעגל לתזמוני פולס. אם `None`, נעשה שימוש ב-`instruction_schedule_map` של ה-QPU.
</details>

### פרמטרים המשמשים להשפיע על אופן פעולת ה-Transpiler
פרמטרים אלה משפיעים על שלבי טרנספילציה ספציפיים. חלקם עשויים להשפיע על שלבים מרובים, אך רוכזו תחת שלב אחד בלבד לצורך פשטות. אם תציין ארגומנט, כגון `initial_layout` עבור ה-Qubit-ים שברצונך להשתמש בהם, הערך הזה עוקף את כל ה-pass-ים שיכלו לשנות אותו. במילים אחרות, ה-Transpiler לא ישנה שום דבר שאתה מציין ידנית. לפרטים על שלבים ספציפיים, ראה [שלבי Transpiler](transpiler-stages).

<details>
  <summary>
    **שלב אתחול**
  </summary>

- `hls_config` (HLSConfig) - מחלקת תצורה אופציונלית `HLSConfig` שמועברת ישירות ל-pass הטרנספורמציה `HighLevelSynthesis`. מחלקת תצורה זו מאפשרת לך לציין רשימות של אלגוריתמי סינתזה והפרמטרים שלהם עבור אובייקטים ברמה גבוהה שונים.
- `init_method` (str) - שם הפלאגין לשימוש בשלב האתחול. כברירת מחדל, לא נעשה שימוש בפלאגין חיצוני. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `list_stage_plugins()` עם `init` עבור ארגומנט שם השלב.
- `unitary_synthesis_method` (str) - שם שיטת סינתזת היחידה לשימוש. כברירת מחדל, נעשה שימוש ב-`default`. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - מילון תצורה אופציונלי שמועבר ישירות לפלאגין סינתזת היחידה. כברירת מחדל הגדרה זו אינה משפיעה מכיוון ששיטת סינתזת היחידה הברירת מחדל אינה מקבלת תצורה מותאמת אישית. החלת תצורה מותאמת אישית נחוצה רק כאשר מצוין פלאגין סינתזת יחידה עם ארגומנט `unitary_synthesis`. מכיוון שזה מותאם אישית לכל פלאגין סינתזת יחידה, עיין בתיעוד הפלאגין כדי לראות כיצד להשתמש באפשרות זו.
</details>

<details>
  <summary>
    **שלב פריסה**
  </summary>

- `initial_layout` (Layout | Dict | List) - מיקום התחלתי של Qubit-ים וירטואליים על Qubit-ים פיזיים. אם פריסה זו הופכת את המעגל לתואם להגבלות `coupling_map`, היא תשמש. הפריסה הסופית אינה מובטחת להיות זהה, מכיוון שה-Transpiler עשוי לסדר מחדש Qubit-ים באמצעות החלפות או אמצעים אחרים. לפרטים מלאים, ראה את [סעיף הפריסה ההתחלתית.](common-parameters#initial-layout)
- `layout_method` (str) - שם pass בחירת פריסה (`default`, `dense`, `sabre`, ו-`trivial`). יכול להיות גם שם הפלאגין החיצוני לשימוש בשלב הפריסה. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `list_stage_plugins()` עם `layout` עבור ארגומנט `stage_name`. ערך ברירת המחדל הוא `sabre`.
</details>

<details>
  <summary>
    **שלב ניתוב**
  </summary>

- `routing_method` (str) - שם pass הניתוב (`basic`, `lookahead`, `default`, `sabre`, או `none`). יכול להיות גם שם הפלאגין החיצוני לשימוש בשלב הניתוב. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `list_stage_plugins()` עם `routing` עבור ארגומנט `stage_name`. ערך ברירת המחדל הוא `sabre`.
</details>

<details>
  <summary>
    **שלב תרגום**
  </summary>

- `translation_method` (str) - שם pass התרגום (`default`, `synthesis`, `translator`, `ibm_backend`, `ibm_dynamic_circuits`, `ibm_fractional`). יכול להיות גם שם הפלאגין החיצוני לשימוש בשלב התרגום. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `list_stage_plugins()` עם `translation` עבור ארגומנט `stage_name`. ערך ברירת המחדל הוא `translator`.
</details>

<details>
  <summary>
    **שלב אופטימיזציה**
  </summary>

- `approximation_degree` (float, בטווח 0-1 | None) - חיוג היוריסטי המשמש לקירוב מעגל (1.0 = ללא קירוב, 0.0 = קירוב מקסימלי). ערך ברירת המחדל הוא 1.0. ציון `None` מגדיר את דרגת הקירוב לשיעור השגיאה המדווח. ראה את [סעיף דרגת הקירוב](common-parameters#approx-degree) לפרטים נוספים.
- `optimization_method` (str) - שם הפלאגין לשימוש בשלב האופטימיזציה. כברירת מחדל לא נעשה שימוש בפלאגין חיצוני. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `list_stage_plugins()` עם `optimization` עבור ארגומנט `stage_name`.
</details>

<details>
  <summary>
    **שלב תזמון**
  </summary>

- `scheduling_method` (str) - שם ה-pass לתזמון. יכול להיות גם שם הפלאגין החיצוני לשימוש בשלב התזמון. ניתן לראות רשימה של פלאגינים מותקנים על ידי הרצת `list_stage_plugins()` עם `scheduling` עבור ארגומנט `stage_name`.
  * 'as_soon_as_possible': תזמן הוראות בצורה חמדנית, מוקדם ככל האפשר על משאב Qubit (כינוי: `asap`).
  * 'as_late_as_possible': תזמן הוראות מאוחר, כלומר, שמור Qubit-ים במצב הבסיסי כאשר אפשרי (כינוי: `alap`). זוהי ברירת המחדל.
</details>

<details>
  <summary>
    **הרצת Transpiler**
  </summary>

- `seed_transpiler` (int) - מגדיר זרעים אקראיים עבור החלקים הסטוכסטיים של ה-Transpiler.
</details>

ערכי ברירת המחדל הבאים משמשים אם לא תציין אף אחד מהפרמטרים לעיל. עיין ב[דף עזר ה-API](../api/qiskit/transpiler_preset) של המתודה למידע נוסף:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>